## Proyecto 2 - Inteligencia Artificial

Jose Angel Morales Farfan - 22689

In [ ]:
import time
import heapq
from collections import deque
import math

# Jerarquía: Arriba, Derecha, Abajo, Izquierda 
DIRECTIONS = [(-1, 0), (0, 1), (1, 0), (0, -1)]

def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def euclidean(a, b):
    return math.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2)

def load_maze(file_path):
    with open(file_path, 'r') as f:
        # Filtramos líneas vacías y cargamos la matriz
        maze = [list(line.strip()) for line in f if line.strip()]
    
    start_nodes = []
    end_nodes = []
    for r in range(len(maze)):
        for c in range(len(maze[0])):
            if maze[r][c] == '2': start_nodes.append((r, c))
            if maze[r][c] == '3': end_nodes.append((r, c))  
    return maze, start_nodes, end_nodes

In [2]:
def reconstruct_path(came_from, start, goal):
    if goal not in came_from: return [] # No hay solución
    current = goal
    path = []
    while current != start:
        path.append(current)
        current = came_from[current]
    return path[::-1]

In [ ]:
def search_uninformed(maze, start, goal, method='BFS'):
    start_time = time.perf_counter()
    frontier = deque([start]) if method == 'BFS' else [start]
    came_from = {start: None}
    explored = set()
    nodes_visited = 0

    while frontier:
        current = frontier.popleft() if method == 'BFS' else frontier.pop()
        nodes_visited += 1
        
        if current == goal:
            break
        
        explored.add(current)
        for dr, dc in DIRECTIONS: # Jerarquía estricta 
            neighbor = (current[0] + dr, current[1] + dc)
            if (0 <= neighbor[0] < len(maze) and 0 <= neighbor[1] < len(maze[0]) and
                maze[neighbor[0]][neighbor[1]] != '1' and neighbor not in came_from):
                came_from[neighbor] = current
                frontier.append(neighbor)

    execution_time = (time.perf_counter() - start_time) * 1000
    return reconstruct_path(came_from, start, goal), nodes_visited, execution_time

In [4]:
def search_informed(maze, start, goal, heuristic_fn, method='A_STAR'):
    start_time = time.perf_counter()
    # (prioridad, actual, costo_g)
    frontier = [(heuristic_fn(start, goal), start, 0)]
    came_from = {start: None}
    cost_so_far = {start: 0}
    nodes_visited = 0

    while frontier:
        _, current, current_cost = heapq.heappop(frontier)
        nodes_visited += 1

        if current == goal:
            break

        for dr, dc in DIRECTIONS:
            neighbor = (current[0] + dr, current[1] + dc)
            if (0 <= neighbor[0] < len(maze) and 0 <= neighbor[1] < len(maze[0]) and
                maze[neighbor[0]][neighbor[1]] != '1'):
                
                new_cost = current_cost + 1
                if neighbor not in cost_so_far or new_cost < cost_so_far[neighbor]:
                    cost_so_far[neighbor] = new_cost
                    # Prioridad: f(n) = g(n) + h(n) para A*, o solo h(n) para Greedy 
                    priority = (new_cost + heuristic_fn(neighbor, goal)) if method == 'A_STAR' else heuristic_fn(neighbor, goal)
                    heapq.heappush(frontier, (priority, neighbor, new_cost))
                    came_from[neighbor] = current

    execution_time = (time.perf_counter() - start_time) * 1000
    return reconstruct_path(came_from, start, goal), nodes_visited, execution_time

In [ ]:
def run_comparison(file_path):
    maze, starts, ends = load_maze(file_path)
    # Usamos el primer punto de inicio y fin como caso base 
    s, e = starts[0], ends[0]
    
    results = []
    # Pruebas No Informadas
    for alg in ['BFS', 'DFS']:
        path, nodes, t = search_uninformed(maze, s, e, alg)
        results.append((alg, len(path), nodes, t))
    
    # Pruebas Informadas
    heuristics = [('Manhattan', manhattan), ('Euclidiana', euclidean)]
    for h_name, h_func in heuristics:
        for alg in ['GREEDY', 'A_STAR']:
            path, nodes, t = search_informed(maze, s, e, h_func, alg)
            results.append((f"{alg} ({h_name})", len(path), nodes, t))

    print(f"{'Algoritmo':<25} | {'Camino':<7} | {'Visitados':<10} | {'Tiempo (ms)':<10}")
    print("-" * 65)
    for res in results:
        print(f"{res[0]:<25} | {res[1]:<7} | {res[2]:<10} | {res[3]:<10.4f}")

run_comparison('test_maze.txt')

Algoritmo                 | Camino  | Visitados  | Tiempo (ms)
-----------------------------------------------------------------
BFS                       | 128     | 665        | 1.7667    
DFS                       | 182     | 565        | 1.2984    
GREEDY (Manhattan)        | 134     | 334        | 1.0980    
A_STAR (Manhattan)        | 128     | 534        | 1.7847    
GREEDY (Euclidiana)       | 130     | 407        | 1.6898    
A_STAR (Euclidiana)       | 128     | 598        | 2.0119    


In [6]:
run_comparison('Prueba_1.txt')

Algoritmo                 | Camino  | Visitados  | Tiempo (ms)
-----------------------------------------------------------------
BFS                       | 119     | 1849       | 6.1115    
DFS                       | 133     | 1580       | 4.8144    
GREEDY (Manhattan)        | 119     | 120        | 1.2109    
A_STAR (Manhattan)        | 119     | 986        | 4.8702    
GREEDY (Euclidiana)       | 137     | 205        | 1.3035    
A_STAR (Euclidiana)       | 119     | 989        | 4.0864    


In [10]:
run_comparison('Laberinto1-1.txt')

Algoritmo                 | Camino  | Visitados  | Tiempo (ms)
-----------------------------------------------------------------
BFS                       | 108     | 10210      | 31.8454   
DFS                       | 1346    | 3728       | 10.2410   
GREEDY (Manhattan)        | 116     | 118        | 0.7248    
A_STAR (Manhattan)        | 108     | 742        | 3.0680    
GREEDY (Euclidiana)       | 112     | 365        | 1.2442    
A_STAR (Euclidiana)       | 108     | 1251       | 3.4546    


In [11]:
run_comparison('Laberinto2-1.txt')

Algoritmo                 | Camino  | Visitados  | Tiempo (ms)
-----------------------------------------------------------------
BFS                       | 108     | 10232      | 57.0811   
DFS                       | 1616    | 5993       | 26.4212   
GREEDY (Manhattan)        | 118     | 121        | 0.5022    
A_STAR (Manhattan)        | 108     | 696        | 2.1366    
GREEDY (Euclidiana)       | 146     | 1553       | 6.9249    
A_STAR (Euclidiana)       | 108     | 1206       | 3.6961    


In [12]:
run_comparison('Laberinto3-1.txt')

Algoritmo                 | Camino  | Visitados  | Tiempo (ms)
-----------------------------------------------------------------
BFS                       | 108     | 9913       | 36.9060   
DFS                       | 602     | 1666       | 7.0040    
GREEDY (Manhattan)        | 116     | 118        | 0.6681    
A_STAR (Manhattan)        | 108     | 645        | 2.4315    
GREEDY (Euclidiana)       | 108     | 139        | 0.5760    
A_STAR (Euclidiana)       | 108     | 1079       | 4.4558    


In [15]:
run_comparison('Laberinto1-2.txt')

Algoritmo                 | Camino  | Visitados  | Tiempo (ms)
-----------------------------------------------------------------
BFS                       | 651     | 4761       | 14.1807   
DFS                       | 1001    | 1121       | 2.9543    
GREEDY (Manhattan)        | 655     | 3336       | 13.9892   
A_STAR (Manhattan)        | 651     | 3736       | 10.0361   
GREEDY (Euclidiana)       | 655     | 2808       | 8.9400    
A_STAR (Euclidiana)       | 651     | 3849       | 11.0848   


In [14]:
run_comparison('Laberinto2-2.txt')

Algoritmo                 | Camino  | Visitados  | Tiempo (ms)
-----------------------------------------------------------------
BFS                       | 315     | 4103       | 12.6750   
DFS                       | 943     | 2405       | 7.0989    
GREEDY (Manhattan)        | 325     | 15265      | 41.5021   
A_STAR (Manhattan)        | 315     | 3299       | 9.8191    
GREEDY (Euclidiana)       | 323     | 3799       | 10.8672   
A_STAR (Euclidiana)       | 315     | 3768       | 51.2731   


In [13]:
run_comparison('Laberinto3-2.txt')

Algoritmo                 | Camino  | Visitados  | Tiempo (ms)
-----------------------------------------------------------------
BFS                       | 183     | 3805       | 12.4682   
DFS                       | 759     | 1957       | 4.9150    
GREEDY (Manhattan)        | 187     | 1841       | 9.0297    
A_STAR (Manhattan)        | 183     | 2040       | 10.0383   
GREEDY (Euclidiana)       | 183     | 941        | 3.6755    
A_STAR (Euclidiana)       | 183     | 2469       | 10.4132   
